# **Unified**

## Setup

In [ ]:
!pip -q uninstall -y transformers tokenizers sentence-transformers bertopic torch torchvision

In [ ]:
# 0.1 Install compatible versions
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q --upgrade pandas requests tqdm sentence-transformers bertopic transformers umap-learn hdbscan pycountry

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.0 requires transformers, which is not installed.
torchtune 0.6.1 requires tokenizers, which is not installed.
torchaudio 2.9.0+cpu requires torch==2.9.0, but you have torch 2.5.1+cu121 which is incompatible.


In [ ]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from transformers import pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 0.2 Imports & paths

import pandas as pd
import numpy as np
import time
import random
import requests
import json
import pycountry

from datetime import date, datetime, timedelta
from tqdm.auto import tqdm

from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/FYP")
RAW_DIR  = BASE_DIR / "data" / "raw"
ENG_DIR  = BASE_DIR / "data" / "engineered"
NLP_DIR  = BASE_DIR / "data" / "nlp"

## Country configuration

In [ ]:
# Destinations for your model
DEST_ISO3 = ["USA", "MYS", "IDN"]

# Origins (given)
ORIG_ISO3 = [
    'AFG','ARG','AUS','BGD','BRA','BRN','CAN','CHN','COL','CUB','DEU',
    'DOM','ECU','EGY','ERI','ESP','ETH','FRA','GBR','GHA','GRC','GTM',
    'GUY','HKG','HND','HTI','IND','IRL','IRN','IRQ','ISR','ITA','JAM',
    'JPN','KEN','KHM','KOR','LAO','LBN','LKA','MEX','MMR','NGA','NIC',
    'NPL','NZL','PAK','PAN','PER','PHL','POL','PRI','PRT','PSE','ROU',
    'RUS','SAU','SDN','SGP','SLV','SOM','SYR','THA','TLS','TTO','TUR',
    'UKR','VEN','VNM','YEM'
]

ALL_FOCUS_ISO3 = sorted(set(DEST_ISO3) | set(ORIG_ISO3))

print("Destinations:", DEST_ISO3)
print("Origins (n):", len(ORIG_ISO3))

Destinations: ['USA', 'MYS', 'IDN']
Origins (n): 70


## GDELT DOC global fetch (conflict + migration)

In [ ]:
START_DATE = date(2025, 1, 1)
END_DATE   = date(2025, 12, 31)
print(f"Global news window: {START_DATE} → {END_DATE}")

GDELT_DOC_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

Global news window: 2025-01-01 → 2025-12-31


In [ ]:
CONFLICT_QUERY = '(war OR conflict OR violence OR "civil unrest" OR fighting OR attack)'
MIGRATION_QUERY = '(migrant OR migrants OR migration OR refugee OR asylum OR immigration OR deportation OR border)'
COOP_QUERY = '(aid OR assistance OR cooperation OR agreement OR treaty OR partnership OR "peace talks" OR summit OR diplomacy OR humanitarian)'

In [ ]:
def _safe_json(resp):
    """
    Some GDELT responses are not JSON (HTML error pages, etc.).
    Return dict or None safely.
    """
    try:
        return resp.json()
    except Exception:
        return None

def fetch_gdelt_docs_global(query, start_date, end_date,
                           max_records=250,
                           step_days=3,
                           max_retries=6,
                           base_sleep=2.0):
    """
    Fetch articles worldwide for a query across a date range.

    Fixes:
    - retries on 429 with exponential backoff and Retry-After
    - avoids crashing on invalid JSON
    - parses datetime from seendatetime OR seendate
    """
    all_rows = []
    current_start = start_date

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=step_days), end_date)

        start_str = current_start.strftime("%Y%m%d%H%M%S")
        end_str   = current_end.strftime("%Y%m%d%H%M%S")

        params = {
            "query": query,
            "mode": "ArtList",
            "maxrecords": int(max_records),
            "format": "json",
            "startdatetime": start_str,
            "enddatetime": end_str,
        }

        attempt = 0
        while attempt <= max_retries:
            try:
                r = requests.get(GDELT_DOC_URL, params=params, timeout=60)

                # Handle 429 rate-limits
                if r.status_code == 429:
                    ra = r.headers.get("Retry-After", None)
                    if ra is not None:
                        sleep_s = float(ra)
                    else:
                        sleep_s = base_sleep * (2 ** attempt) + random.random()
                    print(f"429 rate limit {current_start}–{current_end} | sleeping {sleep_s:.1f}s (attempt {attempt+1})")
                    time.sleep(sleep_s)
                    attempt += 1
                    continue

                r.raise_for_status()

                data = _safe_json(r)
                if data is None:
                    # non-JSON response; backoff and retry
                    sleep_s = base_sleep * (2 ** attempt) + random.random()
                    print(f"Non-JSON response {current_start}–{current_end} | sleeping {sleep_s:.1f}s (attempt {attempt+1})")
                    time.sleep(sleep_s)
                    attempt += 1
                    continue

                arts = data.get("articles", [])
                if arts:
                    all_rows.extend(arts)

                # gentle pacing even on success
                time.sleep(0.6)
                break

            except Exception as e:
                sleep_s = base_sleep * (2 ** attempt) + random.random()
                print(f"Error {current_start}–{current_end}: {e} | sleeping {sleep_s:.1f}s (attempt {attempt+1})")
                time.sleep(sleep_s)
                attempt += 1

        current_start = current_end

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)

    # Parse article datetime properly (avoid "now")
    if "seendatetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["seendatetime"], errors="coerce")
    elif "seendate" in df.columns:
        # many GDELT doc responses include seendate like YYYYMMDDHHMMSS or YYYYMMDD
        df["datetime"] = pd.to_datetime(df["seendate"].astype(str), errors="coerce")
    else:
        df["datetime"] = pd.NaT

    return df

In [ ]:
print("Fetching global CONFLICT news...")
df_conflict_global = fetch_gdelt_docs_global(CONFLICT_QUERY, START_DATE, END_DATE)
df_conflict_global["topic_seed"] = "conflict"

print("Fetching global MIGRATION news...")
df_migration_global = fetch_gdelt_docs_global(MIGRATION_QUERY, START_DATE, END_DATE)
df_migration_global["topic_seed"] = "migration"

print("Fetching global COOP/AID news...")
df_coop_global = fetch_gdelt_docs_global(COOP_QUERY, START_DATE, END_DATE)
df_coop_global["topic_seed"] = "coop"

news_global = pd.concat([df_conflict_global, df_migration_global, df_coop_global], ignore_index=True)
print("Global news shape:", news_global.shape)

news_global.to_csv(NLP_DIR / "gdelt_global_news_raw.csv", index=False)
print("topic_seed counts:")
print(news_global["topic_seed"].value_counts(dropna=False))

print("datetime range:")
print(news_global["datetime"].min(), "→", news_global["datetime"].max())
news_global.head()

Fetching global CONFLICT news...
429 rate limit 2025-01-31–2025-02-03 | sleeping 2.0s (attempt 1)
Fetching global MIGRATION news...
429 rate limit 2025-04-07–2025-04-10 | sleeping 2.3s (attempt 1)
429 rate limit 2025-04-28–2025-05-01 | sleeping 2.3s (attempt 1)
429 rate limit 2025-04-28–2025-05-01 | sleeping 4.7s (attempt 2)
429 rate limit 2025-05-01–2025-05-04 | sleeping 2.7s (attempt 1)
429 rate limit 2025-05-13–2025-05-16 | sleeping 2.2s (attempt 1)
429 rate limit 2025-05-13–2025-05-16 | sleeping 5.0s (attempt 2)
429 rate limit 2025-05-13–2025-05-16 | sleeping 8.9s (attempt 3)
Non-JSON response 2025-07-03–2025-07-06 | sleeping 2.1s (attempt 1)
Non-JSON response 2025-07-03–2025-07-06 | sleeping 4.0s (attempt 2)
Non-JSON response 2025-07-03–2025-07-06 | sleeping 8.3s (attempt 3)
Non-JSON response 2025-07-03–2025-07-06 | sleeping 16.3s (attempt 4)
Non-JSON response 2025-07-03–2025-07-06 | sleeping 32.4s (attempt 5)
Non-JSON response 2025-07-03–2025-07-06 | sleeping 64.3s (attempt 6)
No

,url,url_mobile,title,seendate,socialimage,domain,language,sourcecountry,datetime,topic_seed
0,https://allafrica.com/stories/202501020200.html,,Nigeria : 7 African Countries On the U . S . G...,20250102T143000Z,https://cdn01.allafrica.com/download/pic/main/...,allafrica.com,English,Nigeria,2025-01-02 14:30:00+00:00,conflict
1,https://www.prabhatkhabar.com/opinion/children...,https://www.prabhatkhabar.com/opinion/children...,हिंसा का शिकार होते बच्चे । children victims o...,20250102T063000Z,https://www.prabhatkhabar.com/wp-content/uploa...,prabhatkhabar.com,Hindi,India,2025-01-02 06:30:00+00:00,conflict
2,https://www.elimparcial.com/mundo/2025/01/01/f...,https://www.elimparcial.com/mundo/2025/01/01/f...,Fallecimiento de deportistas en 2024 : Estas f...,20250101T201500Z,https://www.elimparcial.com/resizer/v2/NOGS3Y3...,elimparcial.com,Spanish,Mexico,2025-01-01 20:15:00+00:00,conflict
3,https://www.africanews.com/2025/01/01/sudans-l...,/amp/2025/01/01/sudans-leader-rules-out-pre-wa...,Sudan leader rules out pre - war reconciliatio...,20250101T134500Z,https://static.euronews.com/articles/stories/0...,africanews.com,English,Nigeria,2025-01-01 13:45:00+00:00,conflict
4,https://www.aljazeera.net/news/2025/1/1/%D9%87...,https://www.aljazeera.net/amp/news/2025/1/1/%d...,هجوم على كييف برأس السنة وزيلينسكي يقدم رؤيته ...,20250101T111500Z,https://www.aljazeera.net/wp-content/uploads/2...,aljazeera.net,Arabic,Israel,2025-01-01 11:15:00+00:00,conflict


# Detect discussed country from title text

In [ ]:
# Build a list of (phrase, iso3) for country name detection
country_name_map = []

for iso3 in ALL_FOCUS_ISO3:
    c = pycountry.countries.get(alpha_3=iso3)
    if not c:
        continue
    names = {c.name}
    if hasattr(c, "official_name"):
        names.add(c.official_name)
    # Lower-case
    for n in names:
        country_name_map.append((n.lower(), iso3))

aliases = {
    # AFG
    "afghan": "AFG",
    "afghanistan": "AFG",
    "افغانستان": "AFG",          # Arabic / Persian / Urdu
    "افغان": "AFG",

    # ARG
    "argentina": "ARG",
    "аргентина": "ARG",           # Russian / Ukrainian
    "argentinien": "ARG",         # German
    "argentine": "ARG",           # French

    # AUS
    "australia": "AUS",
    "australie": "AUS",           # French
    "australien": "AUS",          # German
    "австралия": "AUS",           # Russian
    "澳大利亚": "AUS",            # Chinese (simplified)
    "澳大利亞": "AUS",            # Chinese (traditional)

    # BGD
    "bangladesh": "BGD",
    "bangla": "BGD",
    "বাংলাদেশ": "BGD",           # Bengali

    # BRA
    "brazil": "BRA",
    "brasil": "BRA",              # Portuguese / Spanish
    "brésil": "BRA",              # French
    "brasilien": "BRA",           # German
    "брaзилия": "BRA",            # Russian (approx in Latin)
    "бразилия": "BRA",            # Russian Cyrillic

    # BRN
    "brunei": "BRN",
    "بروناي": "BRN",

    # CAN
    "canada": "CAN",
    "canadá": "CAN",              # Spanish / Portuguese
    "kanada": "CAN",              # German / Indonesian form
    "канада": "CAN",              # Russian
    "加拿大": "CAN",              # Chinese

    # CHL (not in ALL_FOCUS_ISO3 but kept)
    "chile": "CHL",
    "chilé": "CHL",
    "чили": "CHL",                # Russian

    # CHN
    "china": "CHN",
    "chine": "CHN",               # French
    "china (land)": "CHN",        # German-style mention
    "kina": "CHN",                # Norwegian / Swedish / Danish
    "中国": "CHN",                # Chinese (simplified)
    "中國": "CHN",                # Chinese (traditional)
    "zhongguo": "CHN",            # Pinyin

    # COL
    "colombia": "COL",
    "colombie": "COL",            # French
    "колумбия": "COL",            # Russian

    # CUB
    "cuba": "CUB",
    "cubá": "CUB",
    "куба": "CUB",                # Russian

    # DEU
    "germany": "DEU",
    "deutschland": "DEU",         # German
    "allemagne": "DEU",           # French
    "alemania": "DEU",            # Spanish
    "alemania federal": "DEU",
    "германия": "DEU",            # Russian
    "niemcy": "DEU",              # Polish

    # DOM
    "dominican republic": "DOM",
    "dominican": "DOM",
    "republica dominicana": "DOM",# Spanish
    "république dominicaine": "DOM", # French
    "доминиканская республика": "DOM", # Russian

    # ECU
    "ecuador": "ECU",
    "ecuatoriana": "ECU",
    "écuador": "ECU",
    "эквадор": "ECU",             # Russian

    # EGY
    "egypt": "EGY",
    "égypte": "EGY",              # French
    "egipto": "EGY",              # Spanish
    "ägypten": "EGY",             # German
    "مصر": "EGY",                 # Arabic (Misr)
    "egypten": "EGY",             # Scandinavian

    # ERI
    "eritrea": "ERI",
    "eritrean": "ERI",
    "إريتريا": "ERI",

    # ESP
    "spain": "ESP",
    "españa": "ESP",              # Spanish
    "espana": "ESP",              # No accent
    "espagne": "ESP",             # French
    "spanien": "ESP",             # German / Scandinavian
    "испания": "ESP",             # Russian (approx Latin)
    "испания": "ESP",             # Russian Cyrillic

    # ETH
    "ethiopia": "ETH",
    "ethiopian": "ETH",
    "etiopía": "ETH",             # Spanish
    "äthiopien": "ETH",           # German
    "إثيوبيا": "ETH",

    # FRA
    "france": "FRA",
    "francia": "FRA",             # Spanish / Italian variant
    "frankreich": "FRA",          # German
    "frança": "FRA",              # Portuguese
    "franța": "FRA",              # Romanian
    "fransa": "FRA",              # Turkish / Arabic transliteration
    "francja": "FRA",             # Polish
    "франция": "FRA",             # Russian

    # GBR
    "britain": "GBR",
    "united kingdom": "GBR",
    "uk": "GBR",
    "great britain": "GBR",
    "royaume-uni": "GBR",         # French
    "reinounido": "GBR",
    "reino unido": "GBR",         # Spanish / Portuguese
    "vereinigtes königreich": "GBR",  # German
    "inglaterra": "GBR",          # Spanish (often used for UK)
    "inghilterra": "GBR",         # Italian
    "англия": "GBR",              # Russian (often used)
    "بريطانيا": "GBR",           # Arabic (Britain)

    # GHA
    "ghana": "GHA",
    "غانا": "GHA",

    # GRC
    "greece": "GRC",
    "gréce": "GRC",
    "grecia": "GRC",              # Spanish / Italian / Romanian
    "grèce": "GRC",               # French
    "griechenland": "GRC",        # German
    "ελλάδα": "GRC",              # Greek
    "ελλαδα": "GRC",
    "grecja": "GRC",              # Polish

    # GTM
    "guatemala": "GTM",
    "guatemalan": "GTM",
    "guatemalá": "GTM",

    # GUY
    "guyana": "GUY",
    "guyanese": "GUY",
    "guyane": "GUY",              # French Guiana style usage

    # HKG
    "hong kong": "HKG",
    "hongkong": "HKG",
    "香港": "HKG",

    # HND
    "honduras": "HND",
    "honduran": "HND",
    "honduraš": "HND",

    # HTI
    "haiti": "HTI",
    "haitian": "HTI",
    "haïti": "HTI",               # French
    "haití": "HTI",               # Spanish

    # IDN
    "indonesia": "IDN",
    "indonesian": "IDN",
    "indonésia": "IDN",           # PT
    "indonésie": "IDN",           # FR
    "indonesien": "IDN",          # DE / DK / NO / SE
    "indonesië": "IDN",           # NL
    "اندونيسيا": "IDN",           # Arabic
    "印度尼西亚": "IDN",           # Chinese
    "印尼": "IDN",                 # Chinese short form
    "indonesia raya": "IDN",

    # IND
    "india": "IND",
    "bharat": "IND",              # Hindi name
    "भारत": "IND",                # Hindi
    "inde": "IND",                # French
    "indien": "IND",              # German / Swedish / Danish
    "индия": "IND",               # Russian
    "الهند": "IND",              # Arabic

    # IRL
    "ireland": "IRL",
    "irlanda": "IRL",             # Spanish / Italian / Romanian
    "irlande": "IRL",             # French
    "irland": "IRL",              # German / Scandinavian
    "ирландия": "IRL",            # Russian

    # IRN
    "iran": "IRN",
    "ایران": "IRN",              # Persian
    "iran (islamic republic of)": "IRN",
    "iranul": "IRN",

    # IRQ
    "iraq": "IRQ",
    "iraqi": "IRQ",
    "irak": "IRQ",                # FR / DE / TR / ES
    "العراق": "IRQ",             # Arabic

    # ISR
    "israel": "ISR",
    "إسرائيل": "ISR",
    "ישראל": "ISR",              # Hebrew

    # ITA
    "italy": "ITA",
    "italia": "ITA",              # Italian / Spanish
    "italie": "ITA",              # French
    "italien": "ITA",             # German
    "итaлия": "ITA",              # Russian approx
    "италия": "ITA",              # Russian

    # JAM
    "jamaica": "JAM",
    "jamaïque": "JAM",
    "جامايكا": "JAM",

    # JPN
    "japan": "JPN",
    "japon": "JPN",               # FR / ES / PT
    "japón": "JPN",
    "japão": "JPN",
    "japan (nippon)": "JPN",
    "日本": "JPN",
    "nihon": "JPN",
    "nippon": "JPN",

    # KEN
    "kenya": "KEN",
    "kenia": "KEN",               # ES / DE
    "كينيا": "KEN",

    # KHM
    "cambodia": "KHM",
    "cambodge": "KHM",            # FR
    "kampuchea": "KHM",
    "kampuchea democrática": "KHM",
    "កម្ពុជា": "KHM",            # Khmer

    # KOR
    "south korea": "KOR",
    "north korea": "KOR",
    "korea": "KOR",
    "republic of korea": "KOR",
    "korean": "KOR",
    "corea del sur": "KOR",       # ES / IT
    "corée du sud": "KOR",        # FR
    "südkorea": "KOR",            # DE
    "대한민국": "KOR",             # Korean
    "한국": "KOR",

    # LAO
    "laos": "LAO",
    "lao": "LAO",
    "lao pdr": "LAO",
    "ລາວ": "LAO",                # Lao

    # LBN
    "lebanon": "LBN",
    "liban": "LBN",               # FR
    "líbano": "LBN",              # ES / PT
    "لبنان": "LBN",

    # LKA
    "sri lanka": "LKA",
    "ceylon": "LKA",
    "ශ්‍රී ලංකා": "LKA",          # Sinhala
    "இலங்கை": "LKA",              # Tamil

    # MEX
    "mexico": "MEX",
    "méxico": "MEX",
    "mexiko": "MEX",              # DE
    "mexique": "MEX",             # FR
    "mexikó": "MEX",
    "мексика": "MEX",             # RU

    # MMR
    "myanmar": "MMR",
    "burma": "MMR",
    "birmania": "MMR",            # ES / IT / PT
    "birmânia": "MMR",
    "ميانمار": "MMR",

    # MYS
    "malaysia": "MYS",
    "malaisie": "MYS",            # FR
    "malasia": "MYS",             # ES
    "malásia": "MYS",             # PT
    "malaysien": "MYS",           # DE
    "ماليزيا": "MYS",             # Arabic
    "马来西亚": "MYS",            # Chinese
    "malaysia (tanah melayu)": "MYS",

    # NGA
    "nigeria": "NGA",
    "nigéria": "NGA",             # FR / PT
    "nigerien": "NGA",
    "نيجيريا": "NGA",

    # NIC
    "nicaragua": "NIC",
    "nicaraguan": "NIC",
    "nicaragüense": "NIC",

    # NPL
    "nepal": "NPL",
    "नेपाल": "NPL",              # Nepali
    "nepál": "NPL",
    "नेपाळ": "NPL",

    # NZL
    "new zealand": "NZL",
    "nouvelle-zélande": "NZL",    # FR
    "neuseeland": "NZL",          # DE
    "zelanda nouă": "NZL",        # RO
    "نیوزیلندا": "NZL",

    # PAK
    "pakistan": "PAK",
    "پاکستان": "PAK",             # Urdu
    "pakistán": "PAK",            # ES

    # PAN
    "panama": "PAN",
    "panamanian": "PAN",
    "panamá": "PAN",

    # PER
    "peru": "PER",
    "perú": "PER",
    "pérou": "PER",               # FR
    "peruan": "PER",
    "перу": "PER",                # RU

    # PHL
    "philippines": "PHL",
    "filipinas": "PHL",           # ES / PT
    "filipijnen": "PHL",          # NL
    "filipine": "PHL",            # Some EU languages
    "الفلبين": "PHL",

    # POL
    "poland": "POL",
    "polish": "POL",
    "polska": "POL",              # Polish
    "pologne": "POL",             # FR
    "polen": "POL",               # DE / SCAND
    "польша": "POL",              # RU

    # PRI
    "puerto rico": "PRI",
    "puertorico": "PRI",
    "porto rico": "PRI",

    # PRT
    "portugal": "PRT",
    "portugalia": "PRT",          # RO / PL style
    "portugalii": "PRT",
    "portugália": "PRT",
    "portugalsko": "PRT",

    # PSE
    "palestine": "PSE",
    "gaza": "PSE",
    "west bank": "PSE",
    "palestina": "PSE",           # Many EU languages
    "فلسطين": "PSE",

    # ROU
    "romania": "ROU",
    "romanian": "ROU",
    "românia": "ROU",
    "roumanie": "ROU",            # FR
    "rumänien": "ROU",            # DE
    "румыния": "ROU",             # RU

    # RUS
    "russia": "RUS",
    "russian federation": "RUS",
    "russland": "RUS",            # DE
    "rusland": "RUS",             # NL / DA
    "russie": "RUS",              # FR
    "rusia": "RUS",               # ES / RO / ID
    "россия": "RUS",              # RU
    "росія": "RUS",               # UA variant usage

    # SAU
    "saudi arabia": "SAU",
    "saudi": "SAU",
    "arabia saudită": "SAU",      # RO
    "arabie saoudite": "SAU",     # FR
    "arabia saudita": "SAU",      # ES / IT
    "saudijska arabija": "SAU",   # Balkan langs
    "المملكة العربية السعودية": "SAU",

    # SDN
    "sudan": "SDN",
    "السودان": "SDN",

    # SGP
    "singapore": "SGP",
    "sgp": "SGP",
    "singapur": "SGP",            # DE / ES / TR etc.
    "singapura": "SGP",           # ID / MS / PT
    "新加坡": "SGP",              # CN

    # SLV
    "el salvador": "SLV",
    "salvadoran": "SLV",
    "salvador": "SLV",

    # SOM
    "somalia": "SOM",
    "الصومال": "SOM",
    "soomaaliya": "SOM",          # Somali

    # SYR
    "syria": "SYR",
    "syrian": "SYR",
    "siria": "SYR",               # ES / IT / PT
    "syrie": "SYR",               # FR
    "سوريا": "SYR",

    # THA
    "thailand": "THA",
    "tailandia": "THA",           # ES / PT
    "thaïlande": "THA",           # FR
    "thailandia": "THA",          # IT
    "thailande": "THA",
    "ประเทศไทย": "THA",          # Thai

    # TLS
    "timor-leste": "TLS",
    "east timor": "TLS",
    "timor oriental": "TLS",
    "timor lorosae": "TLS",

    # TTO
    "trinidad and tobago": "TTO",
    "trinidad": "TTO",
    "trinidad y tobago": "TTO",

    # TUR
    "turkey": "TUR",
    "türkiye": "TUR",
    "tuerkiye": "TUR",
    "turquia": "TUR",             # ES / PT
    "turquie": "TUR",             # FR
    "türkei": "TUR",              # DE
    "турция": "TUR",              # RU

    # UKR
    "ukraine": "UKR",
    "ucrania": "UKR",             # ES
    "ukraina": "UKR",             # Many EU langs
    "ukrajna": "UKR",             # HU
    "украина": "UKR",             # RU
    "україна": "UKR",             # UA

    # USA
    "united states": "USA",
    "u.s.": "USA",
    "u.s": "USA",
    "us ": "USA",
    "usa": "USA",
    "america": "USA",
    "estados unidos": "USA",      # ES / PT
    "ee.uu.": "USA",
    "eeuu": "USA",
    "etats-unis": "USA",          # FR (no accent)
    "états-unis": "USA",
    "vereinigte staaten": "USA",  # DE
    "vereinigten staaten": "USA",
    "stati uniti": "USA",         # IT
    "stati uniti d'america": "USA",
    "stany zjednoczone": "USA",   # PL
    "abd": "USA",                 # TR abbrev
    "amerika birleşik devletleri": "USA",
    "امريكا": "USA",
    "أمريكا": "USA",
    "الولايات المتحدة": "USA",
    "美国": "USA",                # CN (simplified)
    "美國": "USA",                # CN (traditional)
    "amerika serikat": "USA",     # ID
    "amerika syarikat": "USA",    # MS
    "amerika": "USA",             # Many langs

    # VEN
    "venezuela": "VEN",
    "venezuelá": "VEN",
    "vénézuéla": "VEN",
    "венесуэла": "VEN",           # RU

    # VNM
    "vietnam": "VNM",
    "vietnamese": "VNM",
    "việt nam": "VNM",
    "viet nam": "VNM",
    "vietnamul": "VNM",
    "وینام": "VNM",
    "越南": "VNM",                # CN

    # YEM
    "yemen": "YEM",
    "yemeni": "YEM",
    "yémen": "YEM",
    "اليمن": "YEM",
}


for phrase, iso3 in aliases.items():
    country_name_map.append((phrase.lower(), iso3))

def detect_countries_in_text(text):
    if not isinstance(text, str):
        return set()
    t = text.lower()
    hits = set()
    for phrase, iso3 in country_name_map:
        if phrase in t:
            hits.add(iso3)
    return hits

Apply to titles and expand article × country:

In [ ]:
news_global = pd.read_csv(NLP_DIR / "gdelt_global_news_raw.csv")

In [ ]:
# Prepare text (title-based)
df = news_global.copy()
df["title"] = df["title"].fillna("")
df["text"] = df["title"].where(df["title"].str.strip() != "", df["url"].fillna(""))

df = df[df["text"].str.strip() != ""].copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"])

df["month"] = df["datetime"].dt.to_period("M").dt.to_timestamp()

print("After text+language filter:", df.shape)
df[["title", "sourcecountry", "language"]].head()

After text+language filter: (87250, 12)


,title,sourcecountry,language
0,Nigeria : 7 African Countries On the U . S . G...,Nigeria,English
1,हिंसा का शिकार होते बच्चे । children victims o...,India,Hindi
2,Fallecimiento de deportistas en 2024 : Estas f...,Mexico,Spanish
3,Sudan leader rules out pre - war reconciliatio...,Nigeria,English
4,هجوم على كييف برأس السنة وزيلينسكي يقدم رؤيته ...,Israel,Arabic


In [ ]:
rows = []

for idx, row in df.iterrows():
    hits = detect_countries_in_text(row["text"])
    if not hits:
        continue

    for iso3 in hits:
        if iso3 not in ALL_FOCUS_ISO3:
            continue

        rows.append({
            "article_id": idx,
            "country_iso3": iso3,
            "topic_seed": row["topic_seed"],
            "datetime": row["datetime"],
            "month": row["month"],
            "title": row["title"],
            "text": row["text"],
            "language": row.get("language"),
            "sourcecountry": row.get("sourcecountry"),
        })

news_country = pd.DataFrame(rows)
print("Expanded article × country shape:", news_country.shape)
news_country.head()


Expanded article × country shape: (62720, 9)


,article_id,country_iso3,topic_seed,datetime,month,title,text,language,sourcecountry
0,0,NGA,conflict,2025-01-02 14:30:00+00:00,2025-01-01,Nigeria : 7 African Countries On the U . S . G...,Nigeria : 7 African Countries On the U . S . G...,English,Nigeria
1,2,MEX,conflict,2025-01-01 20:15:00+00:00,2025-01-01,Fallecimiento de deportistas en 2024 : Estas f...,Fallecimiento de deportistas en 2024 : Estas f...,Spanish,Mexico
2,2,USA,conflict,2025-01-01 20:15:00+00:00,2025-01-01,Fallecimiento de deportistas en 2024 : Estas f...,Fallecimiento de deportistas en 2024 : Estas f...,Spanish,Mexico
3,3,SDN,conflict,2025-01-01 13:45:00+00:00,2025-01-01,Sudan leader rules out pre - war reconciliatio...,Sudan leader rules out pre - war reconciliatio...,English,Nigeria
4,8,PAK,conflict,2025-01-03 19:30:00+00:00,2025-01-01,Travel to Pakistan restive Kurram district res...,Travel to Pakistan restive Kurram district res...,English,India


# Mark dest vs origin, keep only relevant rows

In [ ]:
news_country["role"] = np.where(
    news_country["country_iso3"].isin(DEST_ISO3),
    "dest",
    np.where(news_country["country_iso3"].isin(ORIG_ISO3), "orig", "other")
)

news_country = news_country[news_country["role"].isin(["dest", "orig"])].copy()
news_country["date"] = news_country["datetime"].dt.date

print(news_country["role"].value_counts())
news_country.head()


role
orig    52368
dest    10352
Name: count, dtype: int64


,article_id,country_iso3,topic_seed,datetime,month,title,text,language,sourcecountry,role,date
0,0,NGA,conflict,2025-01-02 14:30:00+00:00,2025-01-01,Nigeria : 7 African Countries On the U . S . G...,Nigeria : 7 African Countries On the U . S . G...,English,Nigeria,orig,2025-01-02
1,2,MEX,conflict,2025-01-01 20:15:00+00:00,2025-01-01,Fallecimiento de deportistas en 2024 : Estas f...,Fallecimiento de deportistas en 2024 : Estas f...,Spanish,Mexico,orig,2025-01-01
2,2,USA,conflict,2025-01-01 20:15:00+00:00,2025-01-01,Fallecimiento de deportistas en 2024 : Estas f...,Fallecimiento de deportistas en 2024 : Estas f...,Spanish,Mexico,dest,2025-01-01
3,3,SDN,conflict,2025-01-01 13:45:00+00:00,2025-01-01,Sudan leader rules out pre - war reconciliatio...,Sudan leader rules out pre - war reconciliatio...,English,Nigeria,orig,2025-01-01
4,8,PAK,conflict,2025-01-03 19:30:00+00:00,2025-01-01,Travel to Pakistan restive Kurram district res...,Travel to Pakistan restive Kurram district res...,English,India,orig,2025-01-03


## BERTopic on all texts

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

docs = news_country["text"].tolist()
embeddings = embedder.encode(docs, show_progress_bar=True)

topic_model = BERTopic(verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

news_country["topic"] = topics

topic_model.save(str(NLP_DIR / "bertopic_global_model"))

topic_info = topic_model.get_topic_info()
topic_info.to_csv(NLP_DIR / "bertopic_global_topic_info.csv", index=False)
topic_info.head()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1960 [00:00<?, ?it/s]

2026-01-21 19:39:41,292 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-21 19:43:21,834 - BERTopic - Dimensionality - Completed ✓
2026-01-21 19:43:21,841 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-21 19:43:36,822 - BERTopic - Cluster - Completed ✓
2026-01-21 19:43:36,866 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-21 19:43:39,067 - BERTopic - Representation - Completed ✓
2026-01-21 19:43:45,621 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,12892,-1_los_mxico_de_que,"[los, mxico, de, que, unidos, migrantes, estad...",[Condena internacional al régimen militar en B...
1,0,1500,0_iranpressnews_271_cnn_24,"[iranpressnews, 271, cnn, 24, enabel, 233, 180...",[سفير مصر فى بروكسل يلتقى المدير التنفيذى لوكا...
2,1,325,1_abd_ukrayna_ve_rusya,"[abd, ukrayna, ve, rusya, yeni, sava, bir, rus...",[ABD ve Fransa Ukraynada ateşkes çağrısında bu...
3,2,258,2_mignews_fakti_bg_quotidiano,"[mignews, fakti, bg, quotidiano, fatto, forbes...",[Ще избухне ли война между Индия и Пакистан ? ...
4,3,255,3_isw_lb_ua_58,"[isw, lb, ua, 58, bloomberg, nova, 01, aa, dsn...",[Україна надаватиме гуманітарну підтримку пале...


# Label conflict vs migration topics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import re

# --- 1. Load topic_info ---
BASE_DIR = Path("/content/drive/MyDrive/FYP")
NLP_DIR  = BASE_DIR / "data" / "nlp"
tryingts = pd.read_csv(NLP_DIR / "bertopic_global_topic_info.csv")

# --- 2. Define keyword sets (multilingual, high-signal) ---

conflict_kw = {
    # English
    "war","wars","conflict","conflicts","clash","clashes","battle","battles",
    "invasion","invaded","invading","offensive","counteroffensive",
    "bomb","bombs","bombing","bombings","shelling","missile","missiles",
    "rocket","rockets","airstrike","airstrikes","strike","strikes",
    "drone","drones","raid","raids",
    "frontline","frontlines","troop","troops","army","armies","military",
    "forces","hostilities","escalation","kill","killed","deaths",
    "casualties","dead","wounded","massacre","genocide",
    "terror","terrorism","terrorist","terrorists","militia","militias",
    "insurgent","insurgents",
    # Locations often appearing in war contexts (acts as weak conflict prior)
    "gaza","hamas","hezbollah","ukraine","donbas","donbass","kremlin",
    "kharkiv","kyiv","mariupol","crimea","karabakh","nagorno","sudan",
    "darfur","yemen","houthi","huthis",
    # Spanish / Portuguese / Italian
    "guerra","guerras","conflicto","conflictos",
    "ataque","ataques","bombardeo","bombardeos","misil","misiles",
    "ofensiva","alto","fuego","tregua","soldados",
    # French
    "guerre","guerres","conflit","conflits",
    "frappe","frappes","bombardement","bombardements",
    "cessez","feu","cessezlefeu","trêve","soldats",
    # German
    "krieg","konflikt","konflikte","angriff","angriffe",
    "bombardierung","waffe","waffen",
    # Russian-ish latin
    "vojna","voyna","udar","obstrel","obstrely","raketa","rakety","front",
    # Turkish
    "savaş","savasi","savası","çatışma","catisma","ateskes","ateşkes",
    "asker","ordu",
    # Arabic-ish
    "harb","gazza","ghaza",
}

migration_kw = {
    # English
    "migrant","migrants","migration","immigrant","immigrants","immigration",
    "refugee","refugees","asylum","seeker","seekers",
    "relocation","resettlement","integration","deport","deported",
    "deportation","border","borders","crossing","crossings","crossed",
    "cross","landed","landings","boat","boats","dinghy","dinghies",
    "vessel","vessels","ship","ships","smuggler","smugglers",
    "trafficking","camp","camps","shelter","shelters","reception",
    "detention","detained","visa","visas","residence","residency",
    "xenophobia","xenophobic",
    # Mediterranean context
    "mediterranean","mediterraneo","mediterranée","lampedusa",
    "canary","canaries","patera","pateras",
    # Spanish / Portuguese / Italian / French
    "migración","migratorias","inmigración","inmigrante","inmigrantes",
    "refugiado","refugiados","asilo",
    "frontera","fronteras","cruce","cruces","cruzando","cruzan",
    "bote","botes","barco","barcos","lancha","lanchas",
    "deportación","deportaciones","campamento","campamentos",
    "migraçao","migração","imigração","imigrante","imigrantes","refugiados",
    "fronteira","fronteiras",
    "migration","migrants","réfugiés","réfugié","asile","frontière",
    "frontières","bateau","bateaux",
    # German
    "migranten","flüchtling","flüchtlinge","asyl","grenze","grenzen",
    "boot","boote",
    # Turkish
    "göç","goc","göçmen","gocmen","mülteci","multeci",
    "sığınmacı","siginmaci","sığınma","siginma",
    # Arabic-ish
    "lajiin","lajin","hijra","hijrah",
}

coop_kw = {
    # English
    "aid","assistance","support","cooperation","cooperate","agreement","deal","treaty",
    "diplomacy","diplomatic","summit","meeting","partnership","trade","investment",
    "funding","donation","relief","humanitarian","ceasefire","peace","negotiation",
    "talks","pact","collaboration",
    # Indo/Malay common
    "bantuan","kerjasama","kesepakatan","perjanjian","damai","negosiasi",
    # French/Spanish/Portuguese hints
    "aide","coopération","accord","paix","négociation",
    "ayuda","cooperación","acuerdo","paz","negociación",
    "ajuda","cooperação","acordo","paz","negociação",
}

def tokenize(text: str):
    text = str(text).lower()
    return re.split(r"[^a-zA-Záéíóúüñçàèìòùâêîôûäëïöüœæßğışçãõ]+", text)

def score_topic_text(text: str):
    toks = tokenize(text)
    c = sum(t in conflict_kw for t in toks if t)
    co = sum(t in coop_kw for t in toks if t)
    m = sum(t in migration_kw for t in toks if t)
    return c, co, m

conflict_topics, coop_topics, migration_topics, other_topics = [], [], [], []

for _, row in topic_info.iterrows():
    tid = int(row["Topic"])
    if tid == -1:
        continue

    combined = f"{row.get('Name','')} {row.get('Representation','')} {row.get('Representative_Docs','')}"
    c, co, m = score_topic_text(combined)

    # If nothing matched
    if c == 0 and co == 0 and m == 0:
        other_topics.append(tid)
        continue

    # Pick the strongest signal
    mx = max(c, co, m)

    # Tie-break rules (conservative):
    # - if conflict ties with anything -> conflict
    # - else if migration ties with coop -> migration
    # - else coop
    if c == mx and (co == mx or m == mx):
        conflict_topics.append(tid)
    elif m == mx and co == mx:
        migration_topics.append(tid)
    elif c == mx:
        conflict_topics.append(tid)
    elif m == mx:
        migration_topics.append(tid)
    else:
        coop_topics.append(tid)

CONFLICT_TOPICS = sorted(set(conflict_topics))
COOP_TOPICS     = sorted(set(coop_topics))
MIGRATION_TOPICS= sorted(set(migration_topics))

print("Found topics:")
print(" conflict:", len(CONFLICT_TOPICS))
print(" coop    :", len(COOP_TOPICS))
print(" migration:", len(MIGRATION_TOPICS))
print(" other   :", len(set(other_topics)))

def map_topic_to_macro(t):
    if t in CONFLICT_TOPICS:
        return "conflict_topic"
    if t in COOP_TOPICS:
        return "coop_topic"
    if t in MIGRATION_TOPICS:
        return "migration_topic"
    return "other_topic"

news_country["topic_macro"] = news_country["topic"].apply(map_topic_to_macro)
print(news_country["topic_macro"].value_counts(dropna=False))

news_country.to_csv(NLP_DIR / "gdelt_global_news_topics_labelled.csv", index=False)
print("Saved:", NLP_DIR / "gdelt_global_news_topics_labelled.csv")

Found topics:
 conflict: 764
 coop    : 329
 migration: 320
 other   : 364
topic_macro
other_topic        25061
conflict_topic     20532
migration_topic     9036
coop_topic          8091
Name: count, dtype: int64
Saved: /content/drive/MyDrive/FYP/data/nlp/gdelt_global_news_topics_labelled.csv


## Targeted sentiment on conflict + migration topics

In [ ]:
sentiment_pipe = pipeline("sentiment-analysis")

def compute_sentiment(texts, batch_size=32):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        out = sentiment_pipe(batch)
        results.extend(out)
    return results

mask_rel = news_country["topic_macro"].isin(["conflict_topic", "migration_topic"])
texts_rel = news_country.loc[mask_rel, "text"].tolist()

sent_res = compute_sentiment(texts_rel)
sent_df = pd.DataFrame(sent_res)

news_country.loc[mask_rel, "sent_label"] = sent_df["label"].values
news_country.loc[mask_rel, "sent_score"] = sent_df["score"].values

def label_to_signed(label, score):
    if pd.isna(label):
        return np.nan
    if str(label).upper() == "POSITIVE":
        return score
    else:
        return -score

news_country["sent_signed"] = news_country.apply(
    lambda r: label_to_signed(r.get("sent_label"), r.get("sent_score")),
    axis=1
)

news_country[["topic_macro", "sent_signed"]].groupby("topic_macro").describe()


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


  0%|          | 0/924 [00:00<?, ?it/s]

sent_signed                                                    \
                      count      mean       std       min       25%       50%   
topic_macro                                                                     
conflict_topic      20532.0 -0.474307  0.802553 -0.999786 -0.986581 -0.952512   
coop_topic              0.0       NaN       NaN       NaN       NaN       NaN   
migration_topic      9036.0 -0.440113  0.812010 -0.999807 -0.984437 -0.937249   
other_topic             0.0       NaN       NaN       NaN       NaN       NaN   

                                     
                      75%       max  
topic_macro                          
conflict_topic   0.508399  0.999866  
coop_topic            NaN       NaN  
migration_topic  0.633781  0.999800  
other_topic           NaN       NaN

## Aggregation function for monthly NLP drivers

In [ ]:
def aggregate_nlp_drivers(df, country_col):
    # total articles
    total = (
        df.groupby([country_col, "month"])
          .size()
          .rename("n_articles_total")
    )

    # topic counts
    by_topic = (
        df.groupby([country_col, "month", "topic_macro"])
          .size()
          .rename("n_articles")
          .reset_index()
    )

    pivot_topics = (
        by_topic.pivot_table(
            index=[country_col, "month"],
            columns="topic_macro",
            values="n_articles",
            fill_value=0,
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )

    # sentiment averages per macro-topic
    sent = (
        df[~df["sent_signed"].isna()]
          .groupby([country_col, "month", "topic_macro"])["sent_signed"]
          .mean()
          .rename("sent_mean")
          .reset_index()
    )

    sent_pivot = (
        sent.pivot_table(
            index=[country_col, "month"],
            columns="topic_macro",
            values="sent_mean",
            fill_value=np.nan,
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )

    drivers = total.reset_index().merge(pivot_topics, on=[country_col, "month"], how="left")
    drivers = drivers.merge(sent_pivot, on=[country_col, "month"], how="left")

    # ensure topic count columns
    for col in ["conflict_topic", "coop_topic", "migration_topic", "other_topic"]:
        if col not in drivers.columns:
            drivers[col] = 0

    den = drivers["n_articles_total"].replace(0, np.nan)
    drivers["share_conflict_topic"]  = drivers["conflict_topic"]  / den
    drivers["share_coop_topic"]      = drivers["coop_topic"]      / den
    drivers["share_migration_topic"] = drivers["migration_topic"] / den

    return drivers

## Build dest and origin monthly drivers + z-scores

In [ ]:
# Destination side
df_dest = news_country[news_country["role"] == "dest"].copy()
df_dest.rename(columns={"country_iso3": "dest_iso3"}, inplace=True)

nlp_dest_monthly = aggregate_nlp_drivers(df_dest, country_col="dest_iso3")
nlp_dest_monthly["year"] = nlp_dest_monthly["month"].dt.year
nlp_dest_monthly["month_num"] = nlp_dest_monthly["month"].dt.month

# Origin side
df_orig = news_country[news_country["role"] == "orig"].copy()
df_orig.rename(columns={"country_iso3": "orig_iso3"}, inplace=True)

nlp_origin_monthly = aggregate_nlp_drivers(df_orig, country_col="orig_iso3")
nlp_origin_monthly["year"] = nlp_origin_monthly["month"].dt.year
nlp_origin_monthly["month_num"] = nlp_origin_monthly["month"].dt.month

nlp_dest_monthly.head(), nlp_origin_monthly.head()

(  dest_iso3      month  n_articles_total  conflict_topic_x  coop_topic  \
 0       IDN 2025-01-01                37               0.0        10.0   
 1       IDN 2025-02-01                 8               0.0         3.0   
 2       IDN 2025-03-01                 9               1.0         2.0   
 3       IDN 2025-04-01                20               5.0         7.0   
 4       IDN 2025-05-01                14               1.0         7.0   
 
    migration_topic_x  other_topic  conflict_topic_y  migration_topic_y  \
 0                0.0         27.0               NaN                NaN   
 1                0.0          5.0               NaN                NaN   
 2                2.0          4.0          0.503062           0.969551   
 3                0.0          8.0          0.206779                NaN   
 4                0.0          6.0         -0.916954                NaN   
 
    conflict_topic  migration_topic  share_conflict_topic  share_coop_topic  \
 0               

In [ ]:
def zscore_by_country(df, country_col, col):
    return df.groupby(country_col)[col].transform(
        lambda s: (s - s.mean()) / (s.std(ddof=0) or 1.0)
    )

for col in ["share_conflict_topic", "share_coop_topic", "share_migration_topic"]:
    nlp_dest_monthly[f"{col}_z"] = zscore_by_country(nlp_dest_monthly, "dest_iso3", col)
    nlp_origin_monthly[f"{col}_z"] = zscore_by_country(nlp_origin_monthly, "orig_iso3", col)


nlp_dest_monthly.head(), nlp_origin_monthly.head()

(  dest_iso3      month  n_articles_total  conflict_topic_x  coop_topic  \
 0       IDN 2025-01-01                37               0.0        10.0   
 1       IDN 2025-02-01                 8               0.0         3.0   
 2       IDN 2025-03-01                 9               1.0         2.0   
 3       IDN 2025-04-01                20               5.0         7.0   
 4       IDN 2025-05-01                14               1.0         7.0   
 
    migration_topic_x  other_topic  conflict_topic_y  migration_topic_y  \
 0                0.0         27.0               NaN                NaN   
 1                0.0          5.0               NaN                NaN   
 2                2.0          4.0          0.503062           0.969551   
 3                0.0          8.0          0.206779                NaN   
 4                0.0          6.0         -0.916954                NaN   
 
    conflict_topic  migration_topic  share_conflict_topic  share_coop_topic  \
 0               

In [ ]:
nlp_dest_monthly.to_csv(NLP_DIR / "nlp_dest_drivers_monthly.csv", index=False)
nlp_origin_monthly.to_csv(NLP_DIR / "nlp_origin_drivers_monthly.csv", index=False)

print("Saved NLP monthly drivers")

print("DEST coop cols:", [c for c in nlp_dest_monthly.columns if "coop" in c])
print("ORIG coop cols:", [c for c in nlp_origin_monthly.columns if "coop" in c])

print("DEST coop_topic sum:", nlp_dest_monthly.get("coop_topic", pd.Series(dtype=float)).sum())
print("ORIG coop_topic sum:", nlp_origin_monthly.get("coop_topic", pd.Series(dtype=float)).sum())


Saved NLP monthly drivers
DEST coop cols: ['coop_topic', 'share_coop_topic', 'share_coop_topic_z']
ORIG coop cols: ['coop_topic', 'share_coop_topic', 'share_coop_topic_z']
DEST coop_topic sum: 1306.0
ORIG coop_topic sum: 6785.0


In [ ]:
nlp_dest_monthly

,dest_iso3,month,n_articles_total,conflict_topic_x,coop_topic,migration_topic_x,other_topic,conflict_topic_y,migration_topic_y,conflict_topic,migration_topic,share_conflict_topic,share_coop_topic,share_migration_topic,year,month_num,share_conflict_topic_z,share_coop_topic_z,share_migration_topic_z
0,IDN,2025-01-01,37,0.0,10.0,0.0,27.0,NaN,NaN,0,0,0.0,0.270270,0.0,2025,1,0.0,-0.947781,0.0
1,IDN,2025-02-01,8,0.0,3.0,0.0,5.0,NaN,NaN,0,0,0.0,0.375000,0.0,2025,2,0.0,0.009794,0.0
2,IDN,2025-03-01,9,1.0,2.0,2.0,4.0,0.503062,0.969551,0,0,0.0,0.222222,0.0,2025,3,0.0,-1.387098,0.0
3,IDN,2025-04-01,20,5.0,7.0,0.0,8.0,0.206779,NaN,0,0,0.0,0.350000,0.0,2025,4,0.0,-0.218788,0.0
4,IDN,2025-05-01,14,1.0,7.0,0.0,6.0,-0.916954,NaN,0,0,0.0,0.500000,0.0,2025,5,0.0,1.152706,0.0
5,IDN,2025-06-01,9,1.0,3.0,0.0,5.0,-0.985092,NaN,0,0,0.0,0.333333,0.0,2025,6,0.0,-0.371176,0.0
6,IDN,2025-07-01,9,0.0,3.0,0.0,6.0,NaN,NaN,0,0,0.0,0.333333,0.0,2025,7,0.0,-0.371176,0.0
7,IDN,2025-08-01,14,4.0,7.0,0.0,3.0,-0.962763,NaN,0,0,0.0,0.500000,0.0,2025,8,0.0,1.152706,0.0
8,IDN,2025-09-01,18,2.0,9.0,0.0,7.0,0.155126,NaN,0,0,0.0,0.500000,0.0,2025,9,0.0,1.152706,0.0
9,IDN,2025-10-01,13,3.0,4.0,0.0,6.0,-0.888121,NaN,0,0,0.0,0.307692,0.0,2025,10,0.0,-0.605620,0.0
